# Lab 5 (variant) - CNN Image Classification: Legends vs Current Stars

Instead of the glasses dataset, this notebook builds **its own dataset** by downloading football player photos directly inside Colab, then trains a CNN to classify **Legends (old era)** vs **Current stars (new era)**.

**Pipeline (same structure as the original lab):**
1. Install packages
2. Download & build the dataset (new step, replaces Kaggle download)
3. Prepare data (train/val/test split + `ImageFolder`)
4. Create the model (transfer-learning CNN)
5. Train the model
6. Check the result (accuracy, confusion matrix, sample predictions)

> **Note on part 5 of the assignment (Faster R-CNN / Mask R-CNN / RetinaNet):** those are *object detection* models — they need bounding-box annotations (a box around the glasses/object in each image). Scraped player photos have no such annotations, so this notebook does the task as **image classification** (whole photo → class), which is what the original notebook's data pipeline (`ImageFolder`) actually did too. If your teacher specifically wants detection, you'd need a dataset with bounding boxes (e.g. a jersey-number or ball detector) — ask me and I'll set that up separately.

## 1. Install and import necessary packages

In [ ]:
%pip list

In [ ]:
%%capture
!pip install -q torch torchvision matplotlib pillow scikit-learn icrawler

In [ ]:
import os
import shutil
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from icrawler.builtin import BingImageCrawler

from sklearn.metrics import confusion_matrix, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Build the dataset

We search for each player by name and download photos with `icrawler` (uses Bing image search, no API key needed). Feel free to edit the player lists — more players per class = more visual variety = a model that generalizes better (rather than just memorizing one face).

In [ ]:
RAW_DIR = "/content/raw_dataset"
CLASSES = {
    "legends": [
        "Pele football player",
        "Diego Maradona football player",
        "Zinedine Zidane football player",
        "Johan Cruyff football player",
        "Franz Beckenbauer football player",
    ],
    "current": [
        "Lionel Messi football player",
        "Kylian Mbappe football player",
        "Erling Haaland football player",
        "Vinicius Junior football player",
        "Jude Bellingham football player",
    ],
}

IMAGES_PER_PLAYER = 20  # 5 players x 20 = ~100 images per class

if os.path.exists(RAW_DIR):
    shutil.rmtree(RAW_DIR)
os.makedirs(RAW_DIR, exist_ok=True)

for class_name, players in CLASSES.items():
    class_dir = os.path.join(RAW_DIR, class_name)
    os.makedirs(class_dir, exist_ok=True)
    for player in players:
        player_dir = os.path.join(class_dir, player.replace(" ", "_"))
        os.makedirs(player_dir, exist_ok=True)
        crawler = BingImageCrawler(storage={"root_dir": player_dir})
        crawler.crawl(keyword=player, max_num=IMAGES_PER_PLAYER, file_idx_offset=0)
        print(f"Downloaded images for: {player}")

In [ ]:
# Flatten each class folder (move all player subfolder images up one level) and rename uniquely
for class_name in CLASSES:
    class_dir = os.path.join(RAW_DIR, class_name)
    counter = 0
    for player_folder in os.listdir(class_dir):
        player_path = os.path.join(class_dir, player_folder)
        if not os.path.isdir(player_path):
            continue
        for fname in os.listdir(player_path):
            src = os.path.join(player_path, fname)
            ext = os.path.splitext(fname)[1] or ".jpg"
            dst = os.path.join(class_dir, f"{class_name}_{counter}{ext}")
            shutil.move(src, dst)
            counter += 1
        shutil.rmtree(player_path)
    print(f"{class_name}: {counter} images")

In [ ]:
# Remove corrupted / unreadable images (common with scraped data)
def clean_folder(folder):
    removed = 0
    for fname in list(os.listdir(folder)):
        fpath = os.path.join(folder, fname)
        try:
            with Image.open(fpath) as img:
                img.verify()
        except (UnidentifiedImageError, OSError):
            os.remove(fpath)
            removed += 1
    return removed

for class_name in CLASSES:
    n = clean_folder(os.path.join(RAW_DIR, class_name))
    print(f"{class_name}: removed {n} corrupted files")

In [ ]:
# Preview a few images from each class
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, class_name in enumerate(CLASSES):
    class_dir = os.path.join(RAW_DIR, class_name)
    files = os.listdir(class_dir)[:4]
    for col, fname in enumerate(files):
        img = Image.open(os.path.join(class_dir, fname))
        axes[row, col].imshow(img)
        axes[row, col].set_title(class_name)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

## 3. Prepare data: train / val / test split

In [ ]:
SPLIT_DIR = "/content/split_dataset"
TRAIN_RATIO, VAL_RATIO = 0.7, 0.15  # remainder (0.15) goes to test

if os.path.exists(SPLIT_DIR):
    shutil.rmtree(SPLIT_DIR)

random.seed(42)
for class_name in CLASSES:
    src_dir = os.path.join(RAW_DIR, class_name)
    files = os.listdir(src_dir)
    random.shuffle(files)

    n = len(files)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    split_map = {
        "train": files[:n_train],
        "val": files[n_train:n_train + n_val],
        "test": files[n_train + n_val:],
    }

    for split_name, split_files in split_map.items():
        out_dir = os.path.join(SPLIT_DIR, split_name, class_name)
        os.makedirs(out_dir, exist_ok=True)
        for fname in split_files:
            shutil.copy(os.path.join(src_dir, fname), os.path.join(out_dir, fname))

    print(f"{class_name}: train={len(split_map['train'])} val={len(split_map['val'])} test={len(split_map['test'])}")

In [ ]:
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-4
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(os.path.join(SPLIT_DIR, "train"), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(SPLIT_DIR, "val"), transform=eval_transform)
test_dataset = datasets.ImageFolder(os.path.join(SPLIT_DIR, "test"), transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
print("Classes:", class_names)
print("Train/Val/Test sizes:", len(train_dataset), len(val_dataset), len(test_dataset))

## 4. Create the model

We use **transfer learning**: a ResNet18 pretrained on ImageNet, with its final layer replaced for our 2-class problem. This works far better than training a CNN from scratch on ~100 images per class — the pretrained layers already know general visual features (edges, textures, shapes), so we only need to fine-tune the last layers for our task.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze early layers, fine-tune only the later ones
for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(class_names))

model = model.to(device)
print(model.fc)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

## 5. Train the model

In [ ]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    torch.set_grad_enabled(train)
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        if train:
            optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}")

In [ ]:
torch.save(model.state_dict(), "legends_vs_current_resnet18.pth")
print("Model saved.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Check the result: test accuracy, confusion matrix, sample predictions

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
print("Confusion matrix:")
print(cm)

In [ ]:
# Visualize a batch of test predictions
def denormalize(img_tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img_tensor.numpy().transpose(1, 2, 0)
    img = std * img + mean
    return np.clip(img, 0, 1)

images, labels = next(iter(test_loader))
images_gpu = images.to(device)

with torch.no_grad():
    outputs = model(images_gpu)
    probs = torch.softmax(outputs, dim=1)
    preds = outputs.argmax(dim=1).cpu()

n_show = min(8, len(images))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat[:n_show]):
    ax.imshow(denormalize(images[i]))
    true_label = class_names[labels[i]]
    pred_label = class_names[preds[i]]
    confidence = probs[i][preds[i]].item()
    color = "green" if pred_label == true_label else "red"
    ax.set_title(f"true: {true_label}\npred: {pred_label} ({confidence:.2f})", color=color, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Notes

- **Data quality matters more than data size here**: scraped images will include some noise (team photos, fan art, wrong person). If accuracy is lower than expected, manually spot-check `/content/raw_dataset` and delete bad images before re-running the split cell.
- **To try more players / more images**: edit the `CLASSES` dict and `IMAGES_PER_PLAYER` in the dataset-building cell, then re-run from there.
- **To extend toward the lab's part 5 (Faster R-CNN / Mask R-CNN / RetinaNet)**: those need bounding-box labels, not just class folders. A natural detection version of this project would be *"detect the jersey number"* or *"detect the ball"* in a photo — tell me if you want that built out instead.